# Step 7: Prepare and Launch Dashboards


This notebook prepares the files used by the interactive Spatial-VTK dashboards. You will export dashboard-ready Parquet data, create summary tables, and build portable commands for the metrics and QC dashboards.


## Imports

These helpers write dashboard datasets and make readable dashboard table previews.


In [1]:
from spatial_vtk.config.notebook import notebook_timer, register_svtk_cell_timer

with notebook_timer():
    import pandas as pd

    from spatial_vtk.config import SpatialVTKConfig
    from spatial_vtk.io import load_output_table, write_output_table
    from spatial_vtk.visualize.dashboard import (
        display_table,
        write_dashboard_metric_dataset,
        write_dashboard_summary_dataset,
    )
    register_svtk_cell_timer()

Run time: 1.16 s


## Configuration

Load the config and choose dashboard output folders and ports.


In [2]:
from pathlib import Path

# Use the repository root so paths match the public source checkout.
repo_root = Path.cwd()
config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it the active config for later package calls.
cfg = SpatialVTKConfig.from_file(config_path, run_scenario="tutorial").activate()

notebook_overrides = {"metrics_port": 8501, "qc_port": 8502}

Run time: 20.9 ms


## Write Dashboard Datasets

The metrics dashboard reads Parquet files. Summary tables keep common filters and views fast.


In [3]:
# Read the enriched metrics table for the metrics dashboard.
metrics = load_output_table("metrics_enriched")

# Read the QC inventory for the QC dashboard.
qc_summary = load_output_table("qc_inventory")

# Write the dashboard-ready metrics dataset.
write_dashboard_metric_dataset(metrics, partitioned=False)

# Write dashboard summary tables used by the Streamlit app.
write_dashboard_summary_dataset(format="parquet")

# Save the QC trace summary for the QC dashboard.
write_output_table("qc_trace_summary", qc_summary)

metrics.head()

,event_id,station,network,component,model,band,metric,metric_group,period_s,value_obs,...,strike,dip,rake,usgs_url,observed_pickle,event_json,synthetic_mseed,selected_station_count,overlapping_broadband_station_count,event_count
0,ci38038071,BFS,CI,Z,cvmsi,1-2 sec,PGA,amplitude,NaN,0.118,...,312.0,79.0,178.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/synth...,30,52,4.0
1,ci38038071,BHP,CI,Z,cvmsi,1-2 sec,PGV,amplitude,NaN,5.410,...,312.0,79.0,178.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/synth...,30,52,5.0
2,ci38695658,BLC,CI,R,cvmsi,2-3 sec,PSA,spectral,2.0,0.284,...,96.0,48.0,108.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/synth...,30,75,4.0
3,ci39812319,BRE,CI,T,cvmsi,3-5 sec,FAS,spectral,3.0,0.037,...,122.0,85.0,179.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/synth...,30,77,5.0
4,ci38695658,CAC,CI,Z,cvmsi,1-2 sec,PGA,amplitude,NaN,0.096,...,96.0,48.0,108.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/obser...,data/examples/la_basin_five_event_subset/synth...,30,75,NaN


Run time: 41.40 s


## Preview the Metrics Table

The same label lookup used by the dashboards can also make notebook previews easier to read.


In [4]:
# Show a dashboard-style preview with readable metric names, period bands, and column headers.
display_table(
    metrics,
    columns=[
        "event_id",
        "station",
        "component",
        "band",
        "metric",
        "value_obs",
        "value_syn",
        "log2_residual",
        "anderson_2004_gof",
    ],
    max_rows=5,
)

,Event ID,Station,Component,Period Band,Metric,Observed Value,Synthetic Value,Log2 Residual,Anderson 2004 GOF
0,ci38038071,BFS,Z,1-2 sec,Peak acceleration (PGA),0.118,0.104,0.184,8.7
1,ci38038071,BHP,Z,1-2 sec,Peak velocity (PGV),5.410,6.220,-0.201,7.9
2,ci38695658,BLC,R,2-3 sec,Pseudo-spectral acceleration (PSA),0.284,0.211,0.429,8.2
3,ci39812319,BRE,T,3-5 sec,Fourier amplitude spectrum (FAS),0.037,0.052,-0.491,7.4
4,ci38695658,CAC,Z,1-2 sec,Peak acceleration (PGA),0.096,0.089,0.109,8.9


Run time: 10.5 ms


## Build Launch Commands

Run these commands in a terminal when you want to open the interactive dashboards.


In [5]:
# Build portable CLI commands for launching the local Streamlit dashboards.
metrics_command = f"svtk dashboard metrics --port {notebook_overrides['metrics_port']}"
qc_command = f"svtk dashboard qc --port {notebook_overrides['qc_port']}"

print("Metrics dashboard:")
print(metrics_command)
print("\nQC dashboard:")
print(qc_command)

Metrics dashboard:
svtk dashboard metrics --port 8501

QC dashboard:
svtk dashboard qc --port 8502
Run time: 1.0 ms
